In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("test.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
display(df.head())


Dataset shape: (14640, 15)
Columns: ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']

Missing values:
tweet_id                            0
airline_sentiment                   0
airline_sentiment_confidence        0
negativereason                   5462
negativereason_confidence        4118
airline                             0
airline_sentiment_gold          14600
name                                0
negativereason_gold             14608
retweet_count                       0
text                                0
tweet_coord                     13621
tweet_created                       0
tweet_location                   4733
user_timezone                    4820
dtype: int64


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [ ]:
# Show how many examples belong to each sentiment
label_column = "sentiment" if "sentiment" in df.columns else "airline_sentiment"
print(df[label_column].value_counts())
print("\nPercentages:")
print((df[label_column].value_counts(normalize=True) * 100).round(2))


airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64

Percentages:
airline_sentiment
negative    62.69
neutral     21.17
positive    16.14
Name: proportion, dtype: float64


In [ ]:
# Keep only text and sentiment
# This supports both the original dataset and an already-cleaned file.
if "airline_sentiment" in df.columns:
    df = df[["text", "airline_sentiment"]].rename(
        columns={"airline_sentiment": "sentiment"}
    )
else:
    df = df[["text", "sentiment"]].copy()

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


Shape: (14640, 2)
Columns: ['text', 'sentiment']


,text,sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [ ]:
# Check duplicate tweets and examples
print("Duplicate tweets:", df["text"].duplicated().sum())
print("Duplicate text-label pairs:", df.duplicated(subset=["text", "sentiment"]).sum())

# Show a few examples from each sentiment
for label in df["sentiment"].dropna().unique():
    print("\n", label)
    display(df[df["sentiment"] == label].head(3))


Duplicate tweets: 213
Duplicate text-label pairs: 188

 neutral


,text,sentiment
0,@VirginAmerica What @dhepburn said.,neutral
2,@VirginAmerica I didn't today... Must mean I n...,neutral
7,@VirginAmerica Really missed a prime opportuni...,neutral



 positive


,text,sentiment
1,@VirginAmerica plus you've added commercials t...,positive
6,"@VirginAmerica yes, nearly every time I fly VX...",positive
8,"@virginamerica Well, I didn't…but NOW I DO! :-D",positive



 negative


,text,sentiment
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative
5,@VirginAmerica seriously would pay $30 a fligh...,negative


In [ ]:
# Find tweets that have more than one sentiment label
conflict_counts = df.groupby("text")["sentiment"].nunique()
conflicting_texts = conflict_counts[conflict_counts > 1].index

print("Tweets with conflicting sentiment labels:", len(conflicting_texts))
display(df[df["text"].isin(conflicting_texts)].sort_values("text"))


Tweets with conflicting sentiment labels: 25


,text,sentiment
11962,@AmericanAir - keeping AA up in the Air! My cr...,neutral
12123,@AmericanAir - keeping AA up in the Air! My cr...,positive
12130,@AmericanAir @Clarkey_19 we done it with 1 tru...,positive
11969,@AmericanAir @Clarkey_19 we done it with 1 tru...,neutral
11939,@AmericanAir @RobertDwyer AA doesnt charge any...,positive
12100,@AmericanAir @RobertDwyer AA doesnt charge any...,neutral
12047,"@AmericanAir Believe me, I understand. Flight ...",negative
11886,"@AmericanAir Believe me, I understand. Flight ...",positive
12080,@AmericanAir I don't think you should help him...,neutral
11919,@AmericanAir I don't think you should help him...,positive


In [ ]:
# Clean the data
# 1. Remove rows with missing text or sentiment
# 2. Remove every tweet that has conflicting labels
# 3. Remove repeated copies of the same tweet

df_clean = df.dropna(subset=["text", "sentiment"]).copy()
df_clean = df_clean[~df_clean["text"].isin(conflicting_texts)]
df_clean = df_clean.drop_duplicates(subset="text", keep="first")
df_clean = df_clean.reset_index(drop=True)

print("Final dataset shape:", df_clean.shape)
print("\nMissing values:")
print(df_clean.isnull().sum())
print("\nSentiment distribution:")
print(df_clean["sentiment"].value_counts())
print("\nDuplicate tweets remaining:", df_clean["text"].duplicated().sum())

# Confirm no conflicting labels remain
remaining_conflicts = df_clean.groupby("text")["sentiment"].nunique().gt(1).sum()
print("Conflicting tweets remaining:", remaining_conflicts)


Final dataset shape: (14402, 2)

Missing values:
text         0
sentiment    0
dtype: int64

Sentiment distribution:
sentiment
negative    9079
neutral     3046
positive    2277
Name: count, dtype: int64

Duplicate tweets remaining: 0
Conflicting tweets remaining: 0


In [ ]:
# Save to a new file so the original test.csv is not overwritten
df_clean.to_csv("cleaned_sentiment.csv", index=False)

# Check the saved file
check_df = pd.read_csv("cleaned_sentiment.csv")
print("Saved cleaned_sentiment.csv")
print("Shape:", check_df.shape)
print("Columns:", check_df.columns.tolist())
display(check_df.head())


Saved cleaned_sentiment.csv
Shape: (14402, 2)
Columns: ['text', 'sentiment']


,text,sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [ ]:
# Split the cleaned data into training, validation, and test sets
from sklearn.model_selection import train_test_split

X = df_clean["text"]
y = df_clean["sentiment"]

# Keep 20% for the final test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Split the remaining data into 70% train and 10% validation overall
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.125,
    random_state=42, stratify=y_train_val
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))


Training samples: 10080
Validation samples: 1441
Test samples: 2881


In [ ]:
# Calculate class weights using training labels only
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array(["negative", "neutral", "positive"])
weights = compute_class_weight(
    class_weight="balanced", classes=classes, y=y_train
)
class_weight_dict = dict(zip(classes, weights))

print("Class weights:")
for label, weight in class_weight_dict.items():
    print(f"{label}: {weight:.4f}")


Class weights:
negative: 0.5287
neutral: 1.5760
positive: 2.1092


In [ ]:
# Optional: download the cleaned dataset from Google Colab
from google.colab import files
files.download("cleaned_sentiment.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>